# ArthSathi ML — Kaggle Training Notebook

Run each cell in order. Enable GPU T4 in Settings → Accelerator before starting.

In [ ]:
# Cell 1 — Clone repo and setup
!git clone https://github.com/Angela-0001/arthsathi-ml.git
import sys, os
sys.path.insert(0, '/kaggle/working/arthsathi-ml')
os.chdir('/kaggle/working/arthsathi-ml')
print('Ready. GPU:', 'Yes' if __import__('torch').cuda.is_available() else 'No')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q torch sentencepiece tokenizers sentence-transformers sacrebleu faiss-cpu datasets tqdm scipy

In [ ]:
# Cell 3 — Collect and clean data
from data.scripts.collect_schemes import run as collect_schemes
from data.scripts.collect_insurance import run as collect_insurance
from data.scripts.generate_synthetic import run as generate_synthetic
from data.scripts.clean_and_format import run as clean_data

collect_schemes()
collect_insurance()
generate_synthetic()
clean_data()

In [ ]:
# Cell 4 — Download translation corpus (Samanantar from AI4Bharat)
from translation.dataset import download_samanantar, build_plain_text_corpus
download_samanantar('hi', max_samples=100000)
download_samanantar('mr', max_samples=50000)
corpus_path = build_plain_text_corpus()
print('Corpus ready:', corpus_path)

In [ ]:
# Cell 5 — Train our own BPE tokenizer
from translation.tokenizer import train_tokenizer
train_tokenizer('translation/corpus/all_sentences.txt')
print('Tokenizer trained')

In [ ]:
# Cell 6 — Train our own seq2seq translation model (Hindi <-> English)
import argparse
from translation.train import train

args = argparse.Namespace(
    lang='hi', epochs=10, batch_size=64,
    d_model=256, n_heads=4, n_layers=4, max_len=256, lr=1.0
)
train(args)

In [ ]:
# Cell 7 — Train Marathi <-> English
args.lang = 'mr'
args.epochs = 8
train(args)

In [ ]:
# Cell 8 — Build language model dataset
from language_model.build_dataset import run as build_lm_data
build_lm_data()

In [ ]:
# Cell 9 — Train our own language model
import argparse
from language_model.train import train as train_lm

args = argparse.Namespace(
    epochs=5, batch_size=16, d_model=256,
    n_heads=4, n_layers=6, max_len=512, lr=3e-4
)
train_lm(args)

In [ ]:
# Cell 10 — Build recommender FAISS indexes
from models.scheme_recommender.train import run as build_scheme_index
from models.insurance_recommender.train import run as build_insurance_index
build_scheme_index()
build_insurance_index()

In [ ]:
# Cell 11 — Evaluate everything
from translation.evaluate_bleu import evaluate as eval_bleu
from language_model.evaluate import run as eval_lm
from models.scheme_recommender.evaluate import run as eval_schemes
from models.insurance_recommender.evaluate import run as eval_insurance

print('=== Translation BLEU ===')
eval_bleu('hi')
print('\n=== Language Model ===')
eval_lm()
print('\n=== Scheme Recommender ===')
eval_schemes()
print('\n=== Insurance Recommender ===')
eval_insurance()

In [ ]:
# Cell 12 — Save all checkpoints for download
import shutil

shutil.make_archive('/kaggle/working/translation_checkpoints', 'zip',
    '/kaggle/working/arthsathi-ml/translation', 'checkpoints')

shutil.make_archive('/kaggle/working/lm_checkpoints', 'zip',
    '/kaggle/working/arthsathi-ml/language_model', 'checkpoints')

shutil.make_archive('/kaggle/working/recommender_artifacts', 'zip',
    '/kaggle/working/arthsathi-ml/models', '.')

print('Done! Download all 3 zip files from the Output tab on the right.')
print('Unzip into arthsathi-ml/ on your laptop.')